# Задание №2: Построение нейронной сети для машинного перевода (Sequence-to-Sequence)

Данное задание представляет собой пошаговое руководство по созданию модели машинного перевода с использованием архитектуры Seq2Seq на базе LSTM. Вы познакомитесь с основными понятиями обработки естественного языка (NLP), научитесь подготавливать данные, строить и обучать модель, а также использовать её для перевода новых предложений.

---

## 1. Теоретическое введение: ключевые понятия NLP

### 1.1. Корпус (Corpus)
Корпус – это структурированная коллекция текстов, используемая для обучения и оценки моделей NLP. В задаче машинного перевода необходим **параллельный корпус** – набор предложений на исходном языке и соответствующих им переводов на целевой язык.

### 1.2. Токенизация (Tokenization)
Процесс разбиения текста на минимальные единицы – **токены** (слова, знаки препинания, символы). В большинстве задач используют токенизацию на уровне слов. Для этого существует множество инструментов, например `Tokenizer` из Keras.

### 1.3. Эмбеддинги (Embeddings)
Векторные представления слов, которые позволяют нейросети работать с дискретными токенами. Слой **Embedding** преобразует индекс слова в плотный вектор фиксированной размерности, который обучается вместе с моделью.

### 1.4. Seq2Seq модель (Sequence-to-Sequence)
Архитектура нейронной сети, предназначенная для преобразования одной последовательности в другую. Состоит из двух основных компонентов:
- **Энкодер (Encoder)** – читает входную последовательность (предложение на исходном языке) и сжимает её в вектор контекста (скрытое состояние).
- **Декодер (Decoder)** – по вектору контекста генерирует выходную последовательность (перевод), слово за словом.

### 1.5. Рекуррентные нейронные сети (RNN, LSTM, GRU)
RNN – класс сетей, способных обрабатывать последовательности произвольной длины, сохраняя внутреннее состояние. LSTM (Long Short-Term Memory) – разновидность RNN, которая эффективно обучается на длинных последовательностях благодаря механизму вентилей, предотвращающему затухание градиентов.

### 1.6. Инференс (Inference)
Этап использования обученной модели для получения перевода новых предложений. Для Seq2Seq моделей инференс требует отдельной сборки энкодера и декодера, так как во время генерации декодер получает на вход предыдущее сгенерированное слово.

### 1.7. Специальные токены
Для обучения Seq2Seq модели в целевые предложения добавляют токены:
- `<start>` – обозначает начало предложения.
- `<end>` – обозначает конец предложения.

Это позволяет декодеру понимать, когда начать и закончить генерацию.

### 1.8. Функция потерь и метрики
- **Categorical Crossentropy** – стандартная функция потерь для многоклассовой классификации, применяемая к каждому шагу декодера.
- **BLEU (Bilingual Evaluation Understudy)** – метрика качества перевода, оценивающая совпадение n-грамм между гипотезой и эталоном.

---

## 2. Постановка задачи

Вам предоставлен параллельный корпус предложений:
- `russian.txt` – русские фразы (по одной на строку)
- `english.txt` – соответствующие английские переводы (по одной на строку)

Необходимо построить модель Seq2Seq на базе LSTM для перевода с русского на английский.

Если реального корпуса нет, вы можете **сгенерировать синтетические данные** по правилам, описанным ниже.

---

## 3. Структура отчёта

Ваш отчёт должен содержать следующие разделы:

1. **Титульный лист** (название работы, ФИО, группа)
2. **Введение** (цель работы, краткое описание задачи)
3. **Теоретическая часть** (объяснение ключевых понятий: корпус, токенизация, эмбеддинги, Seq2Seq, LSTM, инференс)
4. **Описание данных** (источник данных, статистика: количество предложений, размер словарей, распределение длин)
5. **Подготовка данных** (токенизация, паддинг, создание обучающих выборок, разделение на train/validation)
6. **Построение модели** (архитектура энкодера и декодера, визуализация модели, summary)
7. **Обучение модели** (параметры обучения, графики потерь и точности, анализ переобучения)
8. **Инференс и примеры перевода** (описание процесса генерации, примеры перевода тестовых фраз, анализ качества)
9. **Сохранение и загрузка модели** (код сохранения модели и токенизаторов, пример загрузки и использования)
10. **Выводы** (что получилось, какие были трудности, возможные улучшения)
11. **Список использованных источников**
12. **Приложение** (полный код с комментариями)

---

## 4. Синтетический датасет (если нет реального)

Если у вас нет реального параллельного корпуса, создайте его искусственно. Например:

- Сгенерируйте набор фраз на русском языке (числа, цвета, простые предложения) по шаблонам.
- Создайте соответствующий английский перевод по простым правилам (например, словарь соответствий + порядок слов).

Пример кода для генерации синтетических данных:

```python
import random

def generate_synthetic_data(num_pairs=5000):
    rus_sentences = []
    eng_sentences = []
    
    # Словарь соответствий слов
    word_pairs = {
        'кот': 'cat', 'собака': 'dog', 'мышь': 'mouse', 'дом': 'house',
        'видит': 'sees', 'ловит': 'catches', 'бежит': 'runs',
        'большой': 'big', 'маленький': 'small'
    }
    
    templates_rus = [
        "кот видит собаку",
        "собака бежит в дом",
        "маленький кот ловит мышь",
        "большой дом",
        "мышь бежит",
    ]
    
    for _ in range(num_pairs):
        rus = random.choice(templates_rus)
        # Простейший перевод: замена слов по словарю (без учёта порядка)
        eng = rus
        for ru, en in word_pairs.items():
            eng = eng.replace(ru, en)
        rus_sentences.append(rus)
        eng_sentences.append(eng)
    
    return rus_sentences, eng_sentences

# Генерация датасета
russian_sentences, english_sentences = generate_synthetic_data(5000)
```

Сохраните их в файлы, если хотите:

```python
with open('russian.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(russian_sentences))

with open('english.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(english_sentences))
```

---

## 5. Пошаговое выполнение задания

### 5.1. Подготовка окружения

Создайте новый Notebook и выполните импорт необходимых библиотек:

```python
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import pickle
import random
```

### 5.2. Загрузка и первичный анализ данных

```python
# Загрузка данных (или генерация синтетических)
try:
    with open('russian.txt', encoding='utf-8') as f:
        raw_russian = [line.strip() for line in f if line.strip()]
    with open('english.txt', encoding='utf-8') as f:
        raw_english = [line.strip() for line in f if line.strip()]
except FileNotFoundError:
    # Если файлов нет, генерируем синтетические
    raw_russian, raw_english = generate_synthetic_data(5000)
    print("Сгенерирован синтетический датасет.")

# Уравниваем количество примеров
num_samples = min(len(raw_russian), len(raw_english))
raw_russian = raw_russian[:num_samples]
raw_english = raw_english[:num_samples]

print(f"Загружено пар предложений: {num_samples}")
print(f"Пример: русский: {raw_russian[0]} -> английский: {raw_english[0]}")
```

### 5.3. Токенизация и подготовка последовательностей

```python
# Добавляем маркеры начала и конца к английским предложениям
targets_marked = ['start_ ' + t + ' end_' for t in raw_english]

# Токенизация русского текста (вход)
input_tokenizer = Tokenizer(filters='', lower=True)
input_tokenizer.fit_on_texts(raw_russian)
input_seqs = input_tokenizer.texts_to_sequences(raw_russian)
encoder_input_data = pad_sequences(input_seqs, padding='post')
num_encoder_tokens = len(input_tokenizer.word_index) + 1

# Токенизация английского текста с маркерами (выход)
target_tokenizer = Tokenizer(filters='', lower=True)
target_tokenizer.fit_on_texts(targets_marked)
target_seqs = target_tokenizer.texts_to_sequences(targets_marked)

# Для декодера вход: без последнего слова; целевые данные: без первого слова
decoder_input_data = pad_sequences([s[:-1] for s in target_seqs], padding='post')
decoder_target_data = pad_sequences([s[1:] for s in target_seqs], padding='post')
decoder_target_one_hot = tf.keras.utils.to_categorical(decoder_target_data, 
                                                        num_classes=len(target_tokenizer.word_index)+1)

num_decoder_tokens = len(target_tokenizer.word_index) + 1

print(f"Размер русского словаря: {num_encoder_tokens}")
print(f"Размер английского словаря: {num_decoder_tokens}")
print(f"Форма входных данных энкодера: {encoder_input_data.shape}")
print(f"Форма входных данных декодера: {decoder_input_data.shape}")
print(f"Форма целевых данных (one-hot): {decoder_target_one_hot.shape}")
```

### 5.4. Разделение на обучающую и валидационную выборки

```python
from sklearn.model_selection import train_test_split

# Разделим индексы, так как данные связаны
indices = np.arange(num_samples)
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

encoder_train = encoder_input_data[train_idx]
encoder_val = encoder_input_data[val_idx]
decoder_train_in = decoder_input_data[train_idx]
decoder_val_in = decoder_input_data[val_idx]
decoder_train_out = decoder_target_one_hot[train_idx]
decoder_val_out = decoder_target_one_hot[val_idx]

print(f"Обучающих примеров: {len(train_idx)}")
print(f"Валидационных примеров: {len(val_idx)}")
```

### 5.5. Построение модели Seq2Seq

```python
latent_dim = 256  # размер скрытого состояния

# Энкодер
encoder_inputs = Input(shape=(None,), name='encoder_input')
enc_emb = Embedding(num_encoder_tokens, latent_dim, name='encoder_embedding')(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True, name='encoder_lstm')
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Декодер
decoder_inputs = Input(shape=(None,), name='decoder_input')
dec_emb_layer = Embedding(num_decoder_tokens, latent_dim, name='decoder_embedding')
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name='decoder_lstm')
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax', name='decoder_dense')
decoder_outputs = decoder_dense(decoder_outputs)

# Модель для обучения
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
```

### 5.6. Обучение модели

```python
history = model.fit(
    [encoder_train, decoder_train_in],
    decoder_train_out,
    batch_size=64,
    epochs=30,
    validation_data=([encoder_val, decoder_val_in], decoder_val_out),
    verbose=1
)
```

### 5.7. Визуализация процесса обучения

```python
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Потери')

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title('Точность')
plt.show()
```

### 5.8. Сохранение модели и токенизаторов

```python
# Сохраняем модель в формате .h5
model.save('seq2seq_model.h5')
print("Модель сохранена в seq2seq_model.h5")

# Сохраняем токенизаторы с помощью pickle
with open('input_tokenizer.pickle', 'wb') as f:
    pickle.dump(input_tokenizer, f)
with open('target_tokenizer.pickle', 'wb') as f:
    pickle.dump(target_tokenizer, f)
print("Токенизаторы сохранены.")
```

### 5.9. Построение инференсных моделей

Для перевода новых предложений нам нужны отдельные модели энкодера и декодера. Создадим их:

```python
# Инференс-энкодер
encoder_model = Model(encoder_inputs, encoder_states)

# Инференс-декодер
decoder_state_input_h = Input(shape=(latent_dim,), name='decoder_state_h')
decoder_state_input_c = Input(shape=(latent_dim,), name='decoder_state_c')
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_single_input = Input(shape=(1,), name='decoder_single_input')
dec_single_emb = dec_emb_layer(decoder_single_input)
dec_outputs, state_h, state_c = decoder_lstm(dec_single_emb, initial_state=decoder_states_inputs)
dec_outputs = decoder_dense(dec_outputs)
decoder_model = Model(
    [decoder_single_input] + decoder_states_inputs,
    [dec_outputs] + [state_h, state_c]
)

# Словари для обратного преобразования
reverse_target_word_index = {i: w for w, i in target_tokenizer.word_index.items()}
reverse_source_word_index = {i: w for w, i in input_tokenizer.word_index.items()}
```

### 5.10. Функция перевода

```python
def translate_sentence(input_text, max_len=50):
    # Токенизация входного текста
    input_seq = input_tokenizer.texts_to_sequences([input_text])
    if not input_seq[0]:
        return "Не удалось распознать текст."
    input_seq = pad_sequences(input_seq, maxlen=encoder_input_data.shape[1], padding='post')
    
    # Получение состояний энкодера
    states = encoder_model.predict(input_seq, verbose=0)
    
    # Первый токен декодера - 'start_'
    start_token = target_tokenizer.word_index.get('start_', None)
    if start_token is None:
        return "Токен начала не найден."
    target_seq = np.array([[start_token]])
    
    decoded_sentence = []
    for _ in range(max_len):
        output_tokens, h, c = decoder_model.predict([target_seq] + states, verbose=0)
        # output_tokens имеет форму (1, 1, vocab_size)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        
        # Если встретили токен конца, останавливаемся
        end_token = target_tokenizer.word_index.get('end_', -1)
        if sampled_token_index == end_token:
            break
        
        sampled_word = reverse_target_word_index.get(sampled_token_index, '')
        if sampled_word:
            decoded_sentence.append(sampled_word)
        
        # Обновляем target_seq (следующий вход) и состояния
        target_seq = np.array([[sampled_token_index]])
        states = [h, c]
    
    return ' '.join(decoded_sentence).capitalize()
```

### 5.11. Тестирование перевода

```python
test_sentences = [
    "кот видит собаку",
    "большой дом",
    "мышь бежит"
]

for sent in test_sentences:
    translation = translate_sentence(sent)
    print(f"Исходный: {sent}")
    print(f"Перевод: {translation}\n")
```

### 5.12. Загрузка сохранённой модели и использование (отдельный блок)

Этот блок можно выполнить в новом ноутбуке или после перезапуска ядра, чтобы продемонстрировать загрузку модели из файла.

```python
# Загружаем модель
loaded_model = load_model('seq2seq_model.h5')
print("Модель загружена из seq2seq_model.h5")

# Загружаем токенизаторы
with open('input_tokenizer.pickle', 'rb') as f:
    loaded_input_tokenizer = pickle.load(f)
with open('target_tokenizer.pickle', 'rb') as f:
    loaded_target_tokenizer = pickle.load(f)

# Восстанавливаем архитектуру для инференса (необходимо перестроить инференс-модели)
# Для этого нам нужно знать latent_dim и словари. Можно сохранить параметры,
# но здесь мы просто используем тот же код, что и выше, с загруженными весами.
# Проще всего сохранить и инференс-модели отдельно, но в учебных целях покажем, как воссоздать.

# Извлечём слои из загруженной модели
enc_emb_layer = loaded_model.get_layer('encoder_embedding')
enc_lstm_layer = loaded_model.get_layer('encoder_lstm')
dec_emb_layer = loaded_model.get_layer('decoder_embedding')
dec_lstm_layer = loaded_model.get_layer('decoder_lstm')
dec_dense_layer = loaded_model.get_layer('decoder_dense')

# Строим инференс-энкодер заново
encoder_inputs_inf = Input(shape=(None,))
enc_emb_inf = enc_emb_layer(encoder_inputs_inf)
_, state_h_inf, state_c_inf = enc_lstm_layer(enc_emb_inf)
encoder_model_inf = Model(encoder_inputs_inf, [state_h_inf, state_c_inf])

# Строим инференс-декодер
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_single_input = Input(shape=(1,))
dec_single_emb = dec_emb_layer(decoder_single_input)
dec_outputs, state_h, state_c = dec_lstm_layer(dec_single_emb, initial_state=decoder_states_inputs)
dec_outputs = dec_dense_layer(dec_outputs)
decoder_model_inf = Model(
    [decoder_single_input] + decoder_states_inputs,
    [dec_outputs] + [state_h, state_c]
)

# Словари для обратного преобразования
rev_target = {i: w for w, i in loaded_target_tokenizer.word_index.items()}

# Функция перевода с использованием загруженных моделей
def translate_with_loaded(text, max_len=50):
    seq = loaded_input_tokenizer.texts_to_sequences([text])
    if not seq[0]:
        return "Не распознано"
    seq = pad_sequences(seq, maxlen=encoder_input_data.shape[1], padding='post')
    states = encoder_model_inf.predict(seq, verbose=0)
    target_seq = np.array([[loaded_target_tokenizer.word_index['start_']]])
    decoded = []
    for _ in range(max_len):
        out, h, c = decoder_model_inf.predict([target_seq] + states, verbose=0)
        idx = np.argmax(out[0, -1, :])
        if idx == loaded_target_tokenizer.word_index.get('end_', -1):
            break
        word = rev_target.get(idx, '')
        if word:
            decoded.append(word)
        target_seq = np.array([[idx]])
        states = [h, c]
    return ' '.join(decoded).capitalize()

# Тестируем загруженную модель
for sent in test_sentences:
    print(f"Исходный: {sent}")
    print(f"Перевод (загруженная модель): {translate_with_loaded(sent)}\n")
```

---

## 6. Задания для студентов

### Задание 1. Подготовка данных
- Загрузите файлы `russian.txt` и `english.txt` (или сгенерируйте синтетический датасет).
- Выведите статистику: количество предложений, уникальных слов, среднюю длину предложений.
- Выполните токенизацию и паддинг. Подготовьте данные для обучения: `encoder_input_data`, `decoder_input_data`, `decoder_target_data` (one-hot). Объясните, почему для декодера нужны две последовательности.
- Разделите данные на обучающую и валидационную выборки.

### Задание 2. Построение модели Seq2Seq
- Используя функциональный API Keras, постройте модель с энкодером и декодером (LSTM, latent_dim = 256).
- Скомпилируйте модель с оптимизатором 'adam' и функцией потерь 'categorical_crossentropy'.
- Обучите модель на 30–50 эпохах с валидацией. Используйте `EarlyStopping` для предотвращения переобучения.
- Постройте графики потерь и точности на обучении и валидации.

### Задание 3. Инференс и перевод
- Создайте инференсные модели (энкодер и декодер).
- Реализуйте функцию `translate_sentence(text)`, которая возвращает перевод.
- Протестируйте её на 10 случайных предложениях из тестовой выборки (или на своих примерах). Выведите исходный текст и перевод.

### Задание 4. Сохранение и загрузка модели
- Сохраните обученную модель в формате `.h5`, а также токенизаторы с помощью `pickle`.
- В отдельной ячейке (или новом ноутбуке) загрузите сохранённую модель и токенизаторы, восстановите инференс-модели и выполните перевод нескольких фраз. Убедитесь, что результаты совпадают с полученными ранее.

### Задание 5. Анализ результатов
- Оцените качество перевода визуально. В каких случаях модель ошибается?
- Если есть возможность, рассчитайте BLEU-оценку на тестовой выборке (используйте библиотеку `nltk`).
- Предложите способы улучшения модели (например, увеличение размера скрытого состояния, использование двунаправленного LSTM, добавление механизма внимания).

### Задание 6*. Дополнительно (по желанию)
- Реализуйте механизм внимания (например, с использованием `tf.keras.layers.AdditiveAttention`).
- Сравните качество перевода с базовой моделью.
- Попробуйте использовать предобученные эмбеддинги (FastText, Word2Vec).

---

## 7. Заключение

В ходе выполнения этого задания вы:
- познакомились с основными понятиями NLP;
- научились подготавливать параллельные корпуса для задачи машинного перевода;
- реализовали архитектуру Seq2Seq с LSTM;
- обучили модель и проверили её работу;
- освоили сохранение и загрузку модели для дальнейшего использования.

Полученные навыки являются основой для понимания более сложных архитектур, таких как трансформеры, которые сегодня используются в современных системах перевода (Google Translate, DeepL).